# Exercice 2

In [2]:
from typing import Callable
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk import WordNetLemmatizer
import spacy

nlp = spacy.load("fr_core_news_sm")


def ensure_stopwords_downloaded():
    try:
        nltk.data.find("corpora/stopwords")
    except LookupError:
        nltk.download("stopwords")


def ensure_wordnet_downloaded():
    try:
        nltk.data.find("corpora/wordnet")
    except LookupError:
        nltk.download("wordnet")


ensure_stopwords_downloaded()
ensure_wordnet_downloaded()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Administrateur\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
tweets = [
    "Just finished my breakfast!! Check out https://t.co/xyz #goodMorning ☀️😊",
    "Loving the new #Python3.10 features... performance is amazing!!!",
    "@john_doe The movie was REALLY bad... waste of time :(",
    "Don't miss our special offer!!! Visit our website: https://example.com",
    "I can't believe it's Friday!!! #TGIF #Weekend",
    "Machine Learning & Deep Learning are the future of tech!!! 🚀🚀🚀",
]

### Tâche 1 : Conversion et gestion de base
1. Convertissez tous les tweets en minuscules
2. Supprimez les URLs (identifiez-les avec une regex)
3. Supprimez les mentions (@user) et les hashtags
4. Supprimez les emojis
5. Remplacez les caractères dupliqués excessifs (!!!, ???, ...) par un seul exemplaire

In [4]:
def to_lower(sentence: str) -> str:
    return sentence.lower()

In [5]:
URL_PATTERN = (
    r"\b(?:https?://)?"
    r"(?:www\.)?"
    r"(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}"
    r"(?::\d+)?"
    r"(?:/[^\s?#]*)?"
    r"(?:\?[^\s#]*)?"
    r"(?:#[^\s]*)?"
)


def remove_urls(sentence: str) -> str:
    return re.sub(URL_PATTERN, "", sentence)

In [6]:
USER_PATTERN = r"@\S+"


def remove_users(sentence: str) -> str:
    return re.sub(USER_PATTERN, "", sentence)

In [7]:
HASHTAG_PATTERN = r"#\S+"


def remove_hashtags(sentence: str) -> str:
    return re.sub(HASHTAG_PATTERN, "", sentence)

In [30]:
EMOJI_PATTERN = re.compile(
    "["
    "\U0001f600-\U0001f64f"
    "\U0001f300-\U0001f5ff"
    "\U0001f680-\U0001f6ff"
    "\U0001f700-\U0001f77f"
    "\U0001f780-\U0001f7ff"
    "\U0001f800-\U0001f8ff"
    "\U0001f900-\U0001f9ff"
    "\U0001fa00-\U0001fa6f"
    "\U0001fa70-\U0001faff"
    "\U00002700-\U000027bf"
    "\U00002600-\U000026ff"
    "\u200d"
    "\ufe0f"
    "]+",
    flags=re.UNICODE,
)


def remove_emojis(sentence: str) -> str:
    return EMOJI_PATTERN.sub("", sentence)

In [31]:
PUNCTUATIONS = string.punctuation


def replace_duplicate_punctuations(sentence: str) -> str:
    punctuation_class = "[" + re.escape(PUNCTUATIONS) + "]"
    pattern = re.compile(rf"({punctuation_class})\1+")
    return pattern.sub(r"\1", sentence)

In [32]:
def preprocessing(sentences: list[str], transformers: list[Callable]) -> list[str]:
    sentences_cleaned = []
    for sentence in sentences:
        for transformer in transformers:
            sentence = transformer(sentence)
        sentences_cleaned.append(sentence)
    return sentences_cleaned


tweets_cleaned1 = preprocessing(
    sentences=tweets,
    transformers=[
        to_lower,
        remove_urls,
        remove_users,
        remove_hashtags,
        remove_emojis,
        replace_duplicate_punctuations,
    ],
)
print(tweets_cleaned1)

['just finished my breakfast! check out   ', 'loving the new  features. performance is amazing!', ' the movie was really bad. waste of time :(', "don't miss our special offer! visit our website: ", "i can't believe it's friday!  ", 'machine learning & deep learning are the future of tech! ']


### Tâche 2 : Gestion de la ponctuation et des caractères spéciaux
1. Supprimez la ponctuation inutile, SAUF les apostrophes qui doivent être conservées pour les contractions
2. Supprimez les espaces multiples
3. Gérez les cas particuliers :
   - "can't" doit rester "can't" (contraction)
   - "don't" doit rester "don't"
4. Testez votre logique sur : `"I can't believe... it's AMAZING!!!"`

In [33]:
PUNCTUATIONS = string.punctuation
PUNCTUATIONS_TO_REMOVE = re.sub("'", "", PUNCTUATIONS)


def remove_punctuations(sentence: str) -> str:
    return re.sub(f"[{re.escape(PUNCTUATIONS_TO_REMOVE)}]", " ", sentence)

In [34]:
WHITESPACE_CHARS_PATTERN = r"[\t\n\r\u00A0]+"


def remove_useless_spaces(sentence: str) -> str:
    return re.sub(r"\s{2,}", " ", sentence).strip()

In [35]:
tweets_cleaned2 = preprocessing(
    sentences=tweets_cleaned1,
    transformers=[
        remove_punctuations,
        remove_useless_spaces,
    ],
)
print(tweets_cleaned2)

['just finished my breakfast check out', 'loving the new features performance is amazing', 'the movie was really bad waste of time', "don't miss our special offer visit our website", "i can't believe it's friday", 'machine learning deep learning are the future of tech']


### Tâche 3 : Tokenization et stopwords
1. Tokenisez le texte nettoyé (utilisez NLTK ou une approche simple split)
2. Créez une liste personnalisée de stopwords en français ET en anglais
3. Supprimez les stopwords
4. Gardez les contractions comme "can't", "don't" comme tokens entiers

In [36]:
def tokenize_sentence(sentence: str) -> list[str]:
    return sentence.split(" ")

In [37]:
STOP_WORDS_FR = [
    "et",
    "le",
    "la",
    "les",
    "un",
    "une",
    "de",
    "du",
    "des",
    "en",
    "dans",
    "pour",
    "avec",
    "sans",
    "ce",
    "ces",
    "a",
    "à",
    "il",
    "elle",
    "on",
    "nous",
    "vous",
    "ils",
    "elles",
    "ne",
    "pas",
    "que",
    "qui",
    "se",
    "son",
    "sa",
    "ses",
]

STOP_WORDS_UK = [
    "a",
    "an",
    "the",
    "and",
    "or",
    "but",
    "if",
    "in",
    "on",
    "at",
    "for",
    "with",
    "without",
    "to",
    "of",
    "is",
    "are",
    "was",
    "were",
    "he",
    "she",
    "it",
    "they",
    "we",
    "you",
    "I",
    "me",
    "my",
    "mine",
    "your",
]


def remove_stop_words(tokens: list[str]) -> list[str]:
    return [token for token in tokens if token not in STOP_WORDS_UK]

In [38]:
tweets_cleaned3 = preprocessing(
    sentences=tweets_cleaned2,
    transformers=[
        tokenize_sentence,
        remove_stop_words,
    ],
)
print(tweets_cleaned3)

[['just', 'finished', 'breakfast', 'check', 'out'], ['loving', 'new', 'features', 'performance', 'amazing'], ['movie', 'really', 'bad', 'waste', 'time'], ["don't", 'miss', 'our', 'special', 'offer', 'visit', 'our', 'website'], ['i', "can't", 'believe', "it's", 'friday'], ['machine', 'learning', 'deep', 'learning', 'future', 'tech']]


### Tâche 4 : Lemmatisation
1. Implémentez la lemmatisation avec NLTK WordNetLemmatizer
2. Appliquez-la sur les tokens filtrés
3. Comparez avant/après (montrez des exemples)
4. Optionnel : Utilisez SpaCy pour un résultat amélioré

In [39]:
lemmatizer = WordNetLemmatizer()


def lemmatization_nltk(tokens: list[str]) -> list[str]:
    return [lemmatizer.lemmatize(token) for token in tokens]


def lemmatization_spacy(tokens: list[str]) -> list[str]:
    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc]


tweets_cleaned4 = preprocessing(
    sentences=tweets_cleaned3,
    transformers=[
        lemmatization_nltk,
    ],
)

for before, after in zip(tweets_cleaned3[:3], tweets_cleaned4[:3]):
    print("Avant : ", before)
    print("Après : ", after)
    print()

Avant :  ['just', 'finished', 'breakfast', 'check', 'out']
Après :  ['just', 'finished', 'breakfast', 'check', 'out']

Avant :  ['loving', 'new', 'features', 'performance', 'amazing']
Après :  ['loving', 'new', 'feature', 'performance', 'amazing']

Avant :  ['movie', 'really', 'bad', 'waste', 'time']
Après :  ['movie', 'really', 'bad', 'waste', 'time']



### Tâche 5 : Pipeline complet
1. Créez une fonction `clean_tweet(text)` qui applique TOUTES les étapes précédentes
2. Testez-la sur l'ensemble du corpus
3. Affichez avant/après pour 3 tweets
4. Mesurez la réduction du vocabulaire (nombre de tokens uniques avant/après)

In [40]:
clean_tweet = preprocessing

tweets_cleaned = clean_tweet(
    sentences=tweets,
    transformers=[
        to_lower,
        remove_urls,
        remove_users,
        remove_hashtags,
        remove_emojis,
        replace_duplicate_punctuations,
        remove_punctuations,
        remove_useless_spaces,
        tokenize_sentence,
        remove_stop_words,
        lemmatization_nltk,
    ],
)

for before, after in zip(tweets[:3], tweets_cleaned[:3]):
    nb_tokens_before = len(before.split(" "))
    nb_tokens_after = len(after)
    reduce_rate = abs(nb_tokens_before - nb_tokens_after) / nb_tokens_before * 100
    print("Avant : ", before)
    print("Après : ", after)
    print("Nombre de tokens avant : ", nb_tokens_before)
    print("Nombre de tokens après : ", nb_tokens_after)
    print("Pourcentage d'évolution : ", round(reduce_rate, 2))
    print()

Avant :  Just finished my breakfast!! Check out https://t.co/xyz #goodMorning ☀️😊
Après :  ['just', 'finished', 'breakfast', 'check', 'out']
Nombre de tokens avant :  9
Nombre de tokens après :  5
Pourcentage d'évolution :  44.44

Avant :  Loving the new #Python3.10 features... performance is amazing!!!
Après :  ['loving', 'new', 'feature', 'performance', 'amazing']
Nombre de tokens avant :  8
Nombre de tokens après :  5
Pourcentage d'évolution :  37.5

Avant :  @john_doe The movie was REALLY bad... waste of time :(
Après :  ['movie', 'really', 'bad', 'waste', 'time']
Nombre de tokens avant :  10
Nombre de tokens après :  5
Pourcentage d'évolution :  50.0



## Bonus (Optionnel)

- Implémentez le stemming avec PorterStemmer et comparez avec la lemmatisation
- Gérez les cas spéciaux français : accents, traits d'union
- Créez une classe `TextCleaner` réutilisable avec configuration

In [41]:
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()


def stemming(tokens: list[str]) -> list[str]:
    return [stemmer.stem(token) for token in tokens]


tweets_cleaned_stemming = clean_tweet(
    sentences=tweets,
    transformers=[
        to_lower,
        remove_urls,
        remove_users,
        remove_hashtags,
        remove_emojis,
        replace_duplicate_punctuations,
        remove_punctuations,
        remove_useless_spaces,
        tokenize_sentence,
        remove_stop_words,
        stemming,
    ],
)

In [42]:
print("Comparaison Stemming / Lemmatization\n")

for stem, lemma in zip(tweets_cleaned_stemming[:3], tweets_cleaned[:3]):
    stem = set(stem)
    lemma = set(lemma)
    print("Mots uniquement présent dans le traitement stemming : ", stem - lemma)
    print("Mots uniquement présent dans le traitement lemmatisation : ", lemma - stem)
    print("Mots présent dans les deux traitements : ", stem & lemma)
    print()

Comparaison Stemming / Lemmatization

Mots uniquement présent dans le traitement stemming :  {'finish'}
Mots uniquement présent dans le traitement lemmatisation :  {'finished'}
Mots présent dans les deux traitements :  {'out', 'breakfast', 'check', 'just'}

Mots uniquement présent dans le traitement stemming :  {'love', 'featur', 'perform', 'amaz'}
Mots uniquement présent dans le traitement lemmatisation :  {'performance', 'amazing', 'feature', 'loving'}
Mots présent dans les deux traitements :  {'new'}

Mots uniquement présent dans le traitement stemming :  {'wast', 'movi', 'realli'}
Mots uniquement présent dans le traitement lemmatisation :  {'movie', 'waste', 'really'}
Mots présent dans les deux traitements :  {'time', 'bad'}



In [45]:
class TextCleaner:
    URL_PATTERN = (
        r"\b(?:https?://)?"
        r"(?:www\.)?"
        r"(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}"
        r"(?::\d+)?"
        r"(?:/[^\s?#]*)?"
        r"(?:\?[^\s#]*)?"
        r"(?:#[^\s]*)?"
    )
    USER_PATTERN = r"@\S+"
    HASHTAG_PATTERN = r"#\S+"
    EMOJI_PATTERN = re.compile(
        "["
        "\U0001f600-\U0001f64f"
        "\U0001f300-\U0001f5ff"
        "\U0001f680-\U0001f6ff"
        "\U0001f700-\U0001f77f"
        "\U0001f780-\U0001f7ff"
        "\U0001f800-\U0001f8ff"
        "\U0001f900-\U0001f9ff"
        "\U0001fa00-\U0001fa6f"
        "\U0001fa70-\U0001faff"
        "\U00002700-\U000027bf"
        "\U00002600-\U000026ff"
        "\u200d"
        "\ufe0f"
        "]+",
        flags=re.UNICODE,
    )
    PUNCTUATIONS = string.punctuation
    PUNCTUATIONS_TO_REMOVE = re.sub("'", "", PUNCTUATIONS)
    STOP_WORDS_FR = set(stopwords.words("french"))
    STOP_WORDS_UK = set(stopwords.words("english"))
    STOP_WORDS = STOP_WORDS_FR.union(STOP_WORDS_UK)

    def __init__(self, sentences: list[str]):
        self.sentences = sentences
        self.lemmatizer = WordNetLemmatizer()
        self.stemmer = PorterStemmer()

    def to_lower(self, sentence: str) -> str:
        return sentence.lower()

    def remove_urls(self, sentence: str) -> str:
        return re.sub(self.URL_PATTERN, "", sentence)

    def remove_users(self, sentence: str) -> str:
        return re.sub(self.USER_PATTERN, "", sentence)

    def remove_hashtags(self, sentence: str) -> str:
        return re.sub(self.HASHTAG_PATTERN, "", sentence)

    def remove_emojis(self, sentence: str) -> str:
        return self.EMOJI_PATTERN.sub("", sentence)

    def replace_duplicate_punctuations(self, sentence: str) -> str:
        punctuation_class = "[" + re.escape(self.PUNCTUATIONS) + "]"
        pattern = re.compile(rf"({punctuation_class})\1+")
        return pattern.sub(r"\1", sentence)

    def remove_punctuations(self, sentence: str) -> str:
        return re.sub(f"[{re.escape(self.PUNCTUATIONS_TO_REMOVE)}]", "", sentence)

    def remove_useless_spaces(self, sentence: str) -> str:
        return re.sub(r"\s{2,}", " ", sentence).strip()

    def tokenize_sentence(self, sentence: str) -> list[str]:
        return sentence.split(" ")

    def remove_stop_words(self, tokens: list[str]) -> list[str]:
        return [token for token in tokens if token not in self.STOP_WORDS]

    def lemmatization_nltk(self, tokens: list[str]) -> list[str]:
        return [self.lemmatizer.lemmatize(token) for token in tokens]

    def lemmatization_spacy(self, tokens: list[str]) -> list[str]:
        doc = nlp(" ".join(tokens))
        return [token.lemma_ for token in doc]

    def stemming(self, tokens: list[str]) -> list[str]:
        return [self.stemmer.stem(token) for token in tokens]

    def transform(
        self, sentences: list[str], transformers: list[Callable]
    ) -> list[str]:
        sentences_cleaned = []
        for sentence in sentences:
            for transformer in transformers:
                sentence = transformer(sentence)
            sentences_cleaned.append(sentence)
        return sentences_cleaned

    def fast_transform(self) -> list[str]:
        return self.transform(
            sentences=self.sentences,
            transformers=[
                self.to_lower,
                self.remove_urls,
                self.remove_users,
                self.remove_hashtags,
                self.remove_emojis,
                self.remove_punctuations,
                self.remove_useless_spaces,
                self.tokenize_sentence,
                self.remove_stop_words,
                self.stemming,
            ],
        )


TextCleaner(sentences=tweets).fast_transform()

[['finish', 'breakfast', 'check'],
 ['love', 'new', 'featur', 'perform', 'amaz'],
 ['movi', 'realli', 'bad', 'wast', 'time'],
 ['miss', 'special', 'offer', 'visit', 'websit'],
 ["can't", 'believ', 'friday'],
 ['machin', 'learn', 'deep', 'learn', 'futur', 'tech']]